# Laboratorium: Analiza Matematyczna dla AI
## Temat: Szybka Transformata Fouriera (FFT) w Analizie Szeregów Czasowych

### Wprowadzenie: Po co sztucznej inteligencji Transformata Fouriera?
Większość algorytmów uczenia maszynowego (od regresji po sieci neuronowe) świetnie radzi sobie z prostymi zależnościami, ale kompletnie "ślepnie", gdy dostaje surowe, cykliczne dane zmieniające się w czasie. Rzucenie sieci neuronowej surowego wykresu zużycia prądu z kilku lat skończy się spektakularnym przeuczeniem (*overfitting*) lub chaotycznymi prognozami.

Tutaj z pomocą przychodzi **Analiza Harmoniczna** i jej flagowy algorytm: **FFT (Fast Fourier Transform)**.

Transformata Fouriera pozwala nam przenieść sygnał z **dziedziny czasu** (gdzie widzimy tylko trudne do zinterpretowania wahania) do **dziedziny częstotliwości**. Działa ona jak pryzmat rozszczepiający światło: rozbija skomplikowany, zaszumiony sygnał na sumę prostych, czystych sinusoid o określonych częstotliwościach i amplitudach.

### Cel laboratorium:
W ramach dzisiejszych zajęć wcielisz się w rolę inżyniera AI pracującego dla sektora *Smart Energy*. Twoim zadaniem będzie:
1. **Zbadanie realnych danych** dotyczących godzinowego zapotrzebowania na energię elektryczną w krajowej sieci energetycznej.
2. **Użycie FFT** do bezbłędnego zidentyfikowania ukrytych, matematycznych cykli rządzących ludzkim zachowaniem (dobowych, tygodniowych itp.).


---

### Przygotowanie środowiska
Uruchom poniższą komórkę, aby zaimportować niezbędne biblioteki. Będziemy pracować na standardowym zestawie narzędzi data science: `numpy` do obliczeń numerycznych, `pandas` do manipulacji danymi oraz `matplotlib` do wizualizacji.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Pobieranie danych bezpośrednio z Open Power System Data (czas ładowania: kilka sekund)
# Wybieramy dane godzinowe dla Polski (PL) z lat 2015-2020, które są stabilne i kompletne.
url = "https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv"

print("Pobieranie danych o zużyciu energii elektrycznej w Europie... Proszę czekać.")
# Pobieramy tylko kolumnę z czasem oraz dane dla Polski, aby zaoszczędzić pamięć
df = pd.read_csv(url, usecols=['utc_timestamp', 'PL_load_actual_entsoe_transparency'], parse_dates=['utc_timestamp'])

# 2. Czyszczenie i przygotowanie danych
df.dropna(inplace=True)
df.rename(columns={'utc_timestamp': 'Data', 'PL_load_actual_entsoe_transparency': 'Zuzycie_MW'}, inplace=True)
df.set_index('Data', inplace=True)

# 3. Wycięcie mniejszego fragmentu do analizy (np. pełny rok 2019), aby obliczenia były czytelne
df_year = df.loc['2019-01-01':'2019-12-31'].copy()
sygnal_centrowany = df_year['Zuzycie_MW'] - df_year['Zuzycie_MW'].mean()

# 4. Pierwsza wizualizacja – dziedzina czasu
plt.figure(figsize=(15, 5))
plt.plot(df_year.index, df_year['Zuzycie_MW'], color='royalblue', linewidth=0.8)
plt.title('Godzinowe zużycie energii elektrycznej w Polsce (Rok 2019)')
plt.xlabel('Data')
plt.ylabel('Zapotrzebowanie [MW]')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Pomyślnie załadowano {len(df_year)} obserwacji godzinowych.")
df_year.head()

## Matematyczny fundament: Od DFT do FFT

Naszym celem jest przetransformowanie sygnału z **dziedziny czasu** (ciągu godzinowych odczytów zużycia prądu) do **dziedziny częstotliwości**.

Wynikiem tej operacji jest ciąg liczb zespolonych $X_k$. Każda z nich mówi nam o sile (amplitudzie) i przesunięciu (fazie) konkretnej składowej częstotliwości w naszym sygnale.

Klasyczna **Dyskretna Transformata Fouriera (DFT)** robi to za pomocą wzoru:

$$X_k = \sum_{n=0}^{N-1} x_n e^{-i \frac{2\pi}{N} k n} \quad \text{dla } k = 0, 1, \dots, N-1$$

---

### Gdzie leży problem? Pułapka dwóch pętli $\mathcal{O}(N^2)$

Aby obliczyć **tylko jeden** współczynnik $X_k$, musimy przejść pętlą przez wszystkie $N$ próbek sygnału. A ponieważ takich współczynników musimy policzyć aż $N$, bezpośrednia implementacja tego wzoru wymaga **dwóch zagnieżdżonych pętli `for`**.

To oznacza złożone obliczenia rzędu $\mathcal{O}(N^2)$. Dla analityka danych to katastrofa wydajnościowa:
* Dla jednego roku danych godzinowych ($N = 8760$): to aż **76 milionów** operacji.
* Dla danych minutowych z jednego roku ($N = 525600$): to **276 miliardów** operacji. Komputer zawiesiłby się na długie godziny.

---

### Algorytm Cooleya-Tukeya: Strategia „Dziel i Zwyciężaj”

**Szybka Transformata Fouriera (FFT)** to nie jest nowa matematyka – daje dokładnie ten sam wynik co DFT. To po prostu sprytny algorytm, który drastycznie skraca czas pracy procesora.

Zakładając, że długość naszego sygnału $N$ jest potęgą dwójki, wyobraźmy sobie, że rozbijamy go na dwa mniejsze, niezależne sygnały o długości $N/2$:
1. Sygnał złożony tylko z próbek o indeksach **parzystych** (próbka 0, 2, 4...) – oznaczmy jego transformatę jako $E_k$ (*Even*).
2. Sygnał złożony tylko z próbek o indeksach **nieparzystych** (próbka 1, 3, 5...) – oznaczmy jego transformatę jako $O_k$ (*Odd*).

Rozbijamy sumę DFT na dwie niezależne, mniejsze sumy: jedną dla indeksów **parzystych** ($n = 2m$) oraz jedną dla indeksów **nieparzystych** ($n = 2m + 1$):


$$X_k = \sum_{m=0}^{N/2-1} x_{2m} \cdot e^{-i \frac{2\pi}{N} k (2m)} + \sum_{m=0}^{N/2-1} x_{2m+1} \cdot e^{-i \frac{2\pi}{N} k (2m+1)}$$


Uprośćmy wykładniki potęg, pamiętając, że $\frac{2}{N} = \frac{1}{N/2}$:


$$X_k = \sum_{m=0}^{N/2-1} x_{2m} \cdot e^{-i \frac{2\pi}{N/2} k m} + e^{-i \frac{2\pi}{N} k} \sum_{m=0}^{N/2-1} x_{2m+1} \cdot e^{-i \frac{2\pi}{N/2} k m}$$


Zwróć uwagę na strukturę tych dwóch sum. Każda z nich to nic innego jak **klasyczne równanie DFT, ale obliczane dla sygnału o połowę krótszego ($N/2$)**:

* Pierwsza suma to DFT elementów parzystych sygnału, oznaczmy ją jako $E_k$ (*Even*).

* Druga suma to DFT elementów nieparzystych sygnału, oznaczmy ją jako $O_k$ (*Odd*).

Matematycznie to rozbicie sumy wygląda tak:

$$X_k = E_k + e^{-i \frac{2\pi}{N} k} O_k$$

> **Uwaga:** Element $e^{-i \frac{2\pi}{N} k}$ nazywamy *czynnikiem fazowym* (ang. *twiddle factor*). To po prostu liczba zespolona, która działa jak „łącznik” scalający wyniki dla próbek parzystych i nieparzystych.

---

### Problem brakującej połowy i magia symetrii

Tutaj pojawia się kluczowe pytanie: skoro podzieliliśmy nasz sygnał na dwie mniejsze tablice, to nasze małe transformaty $E_k$ i $O_k$ mają długość tylko $N/2$. Wygenerują one indeksy częstotliwości tylko od $0$ do $N/2 - 1$. Skąd wziąć drugą połowę widma, aby ostateczny wynik miał pełne $N$ elementów i zachował całą informację o sygnale?

Z pomocą przychodzi fundamentalna właściwość matematyki dyskretnej: **widma fal są okresowe (zapętlają się)**.
Jeśli zapytamy małe transformaty $E_k$ i $O_k$ o indeks wykraczający poza ich rozmiar (np. $k + N/2$), one po prostu wrócą na początek i podadzą dokładnie tę samą wartość ($E_{k + N/2} = E_k$).

Co się więc dzieje, gdy w naszym głównym równaniu przeskakujemy do obliczania drugiej połowy widma?
* Składowe $E_k$ i $O_k$ potulnie powtarzają swoje wartości.
* Jedynym elementem całego równania, który „orientuje się”, że przeskoczyliśmy o indeks $N/2$, jest czynnik fazowy. Taki skok oznacza dla niego obrót o równe **180°** na okręgu zespolonym.
* W trygonometrii obrót o 180° (czyli zmiana kierunku na przeciwny) to po prostu **pomnożenie przez -1**.

Gdy połączymy te zjawiska w całość, otrzymujemy ostateczny zestaw dwóch równań, nazywany **strukturą "Motylka" (Butterfly Relation)**, który zaimplementujesz rekurencyjnie:

1. **Dla pierwszej połowy widma ($0 \le k < N/2$):**
   $$X_k = E_k + e^{-i \frac{2\pi}{N} k} O_k$$

2. **Dla drugiej połowy widma (przesuniętej o $N/2$):**
   $$X_{k + N/2} = E_k - e^{-i \frac{2\pi}{N} k} O_k$$

> **Główna korzyść:** Wyliczając skomplikowaną wartość dla pierwszej połowy ($X_k$), **za jednym zamachem i bez żadnych dodatkowych pętli** dostajesz wynik dla drugiej połowy ($X_{k + N/2}$). Wykorzystujesz te same, policzone już bloki $E_k$ oraz $O_k$, zmieniając jedynie znak z plusa na minus!

---

### Podsumowanie: Zysk w kodzie

Dzięki temu, że proces ten powtarzamy rekurencyjnie (dzielimy tablice na pół tak długo, aż zjadą do 1 elementu), redukujemy złożoność obliczeniową do poziomu **$\mathcal{O}(N \log_2 N)$**.

Dla naszych 8192 próbek (najbliższa potęga dwójki dla roku danych):
* Zamiast pierwotnych **67 milionów** operacji w DFT...
* Twoja funkcja FFT wykona ich zaledwie około **106 tysięcy**.

Dzięki tej genialnej sztuczce matematycznej z symetrią Twój kod wykona obliczenia w milisekundach!

## Instrukcja implementacji: FFT Krok po Kroku

Twoim zadaniem jest przetłumaczenie wzorów matematycznych z poprzedniej sekcji na działającą, rekurencyjną funkcję w Pythonie. Algorytm wykorzystujący strategię "dziel i zwyciężaj" wymaga zaprogramowania trzech głównych faz: warunku stopu, podziału danych oraz syntezy wyników (motylka).

Postępuj zgodnie z poniższymi instrukcjami, aby poprawnie uzupełnić kod.

### Krok 1: Warunek stopu rekurencji (Base Case)
Każda funkcja rekurencyjna musi wiedzieć, kiedy przestać wywoływać samą siebie. Zastanów się, jaki jest najmniejszy, niepodzielny problem w naszym algorytmie.
* Najmniejszym możliwym podproblemem jest wektor zawierający dokładnie $1$ element.
* Transformata DFT dla sygnału jednoelementowego $x = [x_0]$ to po prostu ten sam wektor $X = [x_0]$.
* Zmierz długość przekazanego wektora wejściowego. Jeśli długość wynosi $1$ (lub mniej), natychmiast zwróć ten wektor.

### Krok 2: Podział sygnału (Divide)
Jeśli wektor jest dłuższy niż $1$ element, musisz go rozbić na dwie niezależne połowy.
* Stwórz zmienną przechowującą elementy z indeksów parzystych wektora wejściowego ($x_0, x_2, x_4, \dots$).
* Stwórz zmienną przechowującą elementy z indeksów nieparzystych wektora wejściowego ($x_1, x_3, x_5, \dots$).
* Wykorzystaj potężne możliwości biblioteki NumPy – użyj wycinania (*slicing*) z odpowiednim krokiem (np. `[::2]`).

### Krok 3: Wywołania rekurencyjne (Conquer)
Mając już podzielony sygnał, musisz zlecić algorytmowi wykonanie obliczeń dla obu połówek.
* Wywołaj swoją własną funkcję dla tablicy elementów parzystych i zapisz wynik jako $E_k$.
* Wywołaj swoją własną funkcję dla tablicy elementów nieparzystych i zapisz wynik jako $O_k$.

### Krok 4: Połączenie wyników (Combine / Butterfly Effect)
To najważniejszy etap, w którym dzieje się cała matematyczna magia, a obie obliczone połówki łączą się w pełne widmo.
* Zainicjalizuj wektor wynikowy (np. za pomocą funkcji `np.zeros()`). Musi mieć taką samą długość co oryginalny wektor wejściowy. Zadbaj o to, aby wymusić dla niego typ zespolony `complex`, inaczej Python obetnie ułamki urojone!
* Użyj pętli, aby iterować po indeksie $k$ od $0$ do $N/2 - 1$.
* Wewnątrz pętli oblicz czynnik fazowy dla danego kroku $k$: jest to element $O_k$ pomnożony przez $e^{-i \frac{2\pi}{N} k}$. Pamiętaj, że w Pythonie jednostkę urojoną $i$ zapisujemy jako `1j`, a do eksponenty wykorzystujemy funkcję `np.exp()`.
* Wypełnij pierwszą połowę wektora wynikowego: pod indeksem $k$ zapisz sumę $E_k$ oraz czynnika fazowego.
* Wypełnij drugą połowę wektora wynikowego: pod indeksem $k + N/2$ zapisz różnicę $E_k$ oraz czynnika fazowego.
* Zwróć gotowy wektor wynikowy kończąc tym samym działanie funkcji.

In [ ]:
import numpy as np

def moja_fft(x):
    """
    Rekurencyjna implementacja algorytmu FFT (Cooley-Tukey).
    Argument:
        x: np.array liczb rzeczywistych lub zespolonych o długości N (będącej potęgą 2)
    Zwraca:
        np.array liczb zespolonych (widmo sygnału)
    """
   #TODO

## Część 3: Test weryfikacyjny na sygnale syntetycznym

Zanim przeanalizujemy rzeczywiste zapotrzebowanie na prąd, musimy upewnić się, że Twoja funkcja `moja_fft` potrafi bezbłędnie zidentyfikować składowe sygnału, który znamy na wylot.

Stworzymy teraz **sygnał syntetyczny**. Będzie to suma dwóch fal sinusoidalnych o różnych, znanych nam z góry częstotliwościach:
1. Fala wolna o częstotliwości $f_1 = 5\text{ Hz}$ i amplitudzie $1.0$
2. Fala szybka o częstotliwości $f_2 = 20\text{ Hz}$ i amplitudzie $0.5$

Równanie naszego testowego sygnału wygląda tak:
$$y(t) = \sin(2\pi \cdot 5 \cdot t) + 0.5 \cdot \sin(2\pi \cdot 20 \cdot t)$$

Ponieważ zaimplementowany przez Ciebie algorytm wymaga, aby liczba próbek $N$ była potęgą dwójki, wygenerujemy dokładnie $N = 128$ próbek w czasie 1 sekundy.
Jeśli napisałeś funkcję bezbłędnie, na prawym wykresie powinieneś zobaczyć dokładnie dwa ostre piki: jeden na wartości 5, a drugi na wartości 20.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Parametry sygnału syntetycznego
N = 128             # Liczba próbek (musi być potęgą dwójki!)
fs = 128            # Częstotliwość próbkowania (128 próbek na sekundę)
t = np.linspace(0, 1, N, endpoint=False)  # Wektor czasu (równo 1 sekunda)

# 2. Generowanie sygnału: suma dwóch sinusoid (5 Hz i 20 Hz)
f1, f2 = 5, 20
sygnal_syntetyczny = np.sin(2 * np.pi * f1 * t) + 0.5 * np.sin(2 * np.pi * f2 * t)

# 3. OBLICZENIA - Wykorzystujemy Twoją funkcję!
widmo = moja_fft(sygnal_syntetyczny)

# 4. Przygotowanie wektora częstotliwości i odcięcie ujemnej połowy widma
czestotliwosci = np.fft.fftfreq(N, 1/fs)
polowa = N // 2

czestotliwosci_dodatnie = czestotliwosci[:polowa]
# Aby uzyskać rzeczywistą amplitudę, dzielimy moduł przez (N/2)
amplitudy = np.abs(widmo[:polowa]) / (N / 2)

# 5. Wizualizacja wyników
plt.figure(figsize=(14, 5))

# Wykres 1: Dziedzina czasu (to widzi ludzkie oko)
plt.subplot(1, 2, 1)
plt.plot(t, sygnal_syntetyczny, color='teal', marker='.', linestyle='-')
plt.title('Sygnał w dziedzinie czasu')
plt.xlabel('Czas [s]')
plt.ylabel('Amplituda')
plt.grid(True, alpha=0.3)

# Wykres 2: Dziedzina częstotliwości (to widzi algorytm FFT)
plt.subplot(1, 2, 2)
plt.stem(czestotliwosci_dodatnie, amplitudy, basefmt=" ")
plt.title('Widmo sygnału (Wynik z Twojej funkcji)')
plt.xlabel('Częstotliwość [Hz]')
plt.ylabel('Amplituda')
plt.xlim(0, 30) # Skupiamy się na zakresie do 30 Hz
plt.xticks(np.arange(0, 31, 5))
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Część 4: Analiza rzeczywistych danych – Detektywistyka w sieci energetycznej

Twoja funkcja `moja_fft` przeszła pomyślnie testy na sterylnym sygnale syntetycznym. Czas rzucić ją na głęboką wodę i sprawdzić, jak poradzi sobie z potężnym, zaszumionym zbiorem danych z krajowej sieci elektroenergetycznej.

Za chwilę użyjesz swojej implementacji, aby z surowego potoku liczb wyciągnąć matematyczny dowód na istnienie konkretnych rytmów w życiu milionów ludzi.

---

### Instrukcja wykonania zadania:

W kolejnej komórce kodowej (którą znajdziesz poniżej) musisz zaprogramować pełną ścieżkę analizy sygnału. Postępuj według następujących kroków:

#### Krok 1: Przycięcie danych do potęgi dwójki
Nasz zaimplementowany algorytm wymaga, aby liczba próbek $N$ była potęgą 2. Pełny rok danych ma 8760 godzin. Najbliższą mniejszą potęgą dwójki jest **$2^{13} = 8192$**.
* **Twoje zadanie:** Odetnij pierwsze 8192 próbki ze zmiennej `sygnal_centrowany` (czyli sygnału z odjętą średnią), którą przygotowaliśmy na początku laboratorium.

#### Krok 2: Obliczenie transformaty i wycięcie dodatniej części widma
* Przepuść przycięty sygnał przez swoją funkcję `moja_fft`.
* Wygeneruj wektor częstotliwości za pomocą `np.fft.fftfreq(N, d=1.0)` (ponieważ nasz krok próbkowania $d$ wynosi dokładnie 1 godzinę).
* Ponieważ widmo jest symetryczne, weź tylko pierwszą połowę częstotliwości (indeksy od `0` do `N//2`).
* Oblicz **widmo gęstości mocy**, podnosząc moduł (wartość bezwzględną) wyniku transformaty do kwadratu: $|X(f)|^2$.

#### Krok 3: Najważniejsze – Zamiana częstotliwości na ludzkie okresy ($T$)
Surowe częstotliwości zwrócone przez algorytm mają abstrakcyjną jednostkę $[1/\text{godzina}]$. Przykładowo, wartość `0.04166` niewiele nam mówi o ludzkich zachowaniach.
* **Twoje zadanie:** Aby wykres był w pełni zrozumiały, przelicz częstotliwości dodatnie na **okres (czas trwania jednego pełnego cyklu) wyrażony w godzinach**.
* Wykorzystaj fundamentalny wzór fizyczny:
  $$T = \frac{1}{f}$$
* *Wskazówka:* Uważaj na pierwszy element wektora częstotliwości ($f = 0$). Dzielenie przez zero wygeneruje błąd, dlatego dla celów wizualizacji możesz zacząć rysowanie wykresu od drugiego elementu (indeks `1:`).

#### Krok 4: Wizualizacja i analiza detektywistyczna
Narysuj wykres, na którym osią $X$ będzie obliczony okres $T$ (w godzinach), a osią $Y$ moc sygnału. Ogranicz oś $X$ (za pomocą `plt.xlim(0, 200)`), ponieważ interesują nas zjawiska zachodzące w przeciągu kilku dni.

---

### Czego masz szukać na wykresie?
Jeśli wszystko wykonałeś poprawnie, na wykresie zobaczysz kilka gigantycznych, dominujących "pików" (linii strzelających w górę). Twoim zadaniem po uruchomieniu kodu będzie zidentyfikowanie wartości na osi $X$ dla trzech najwyższych punktów.

Zastanów się i zapisz wnioski:
1. Jaki okres (ile godzin) reprezentuje najwyższy pik? Z jakiego ludzkiego nawyku on wynika?
2. Co oznacza drugi pod względem wysokości pik i dlaczego wynosi dokładnie połowę pierwszego?
3. Gdzie znajduje się trzeci wyraźny pik (w okolicach ilu godzin) i co reprezentuje?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Krok 1: Przycięcie danych do potęgi dwójki (2^13 = 8192)
N = 8192
sygnal_przyciety = sygnal_centrowany[:N]

# Krok 2: Obliczenie transformaty i wycięcie dodatniej części widma
print("Obliczanie FFT dla rzeczywistych danych... To potrwa ułamek sekundy!")
widmo_rzeczywiste = moja_fft(sygnal_przyciety)
czestotliwosci = np.fft.fftfreq(N, d=1.0)

polowa = N // 2
czestotliwosci_dodatnie = czestotliwosci[:polowa]
widmo_dodatnie = widmo_rzeczywiste[:polowa]

# Obliczenie widma gęstości mocy
widmo_mocy = np.abs(widmo_dodatnie) ** 2

# Krok 3: Zamiana częstotliwości na ludzkie okresy (T)
# Pomijamy pierwszy indeks [0], aby uniknąć dzielenia przez zero (częstotliwość f = 0)
okresy_T = 1.0 / czestotliwosci_dodatnie[1:]
moc_wykres = widmo_mocy[1:]

# Krok 4: Wizualizacja i analiza detektywistyczna
plt.figure(figsize=(15, 6))

# Rysujemy wykres
plt.plot(okresy_T, moc_wykres, color='darkorange', linewidth=1.5)

plt.title('Spektrogram zużycia energii elektrycznej w Polsce (Okresy zjawisk)')
plt.xlabel('Okres trwania jednego cyklu (T) [godziny]')
plt.ylabel('Moc składowej sygnału')

# Skupiamy się na zakresie do 200 godzin (nieco ponad tydzień)
plt.xlim(0, 200)
plt.xticks(np.arange(0, 201, 12)) # Podziałka co 12 godzin

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Część 5: Inżynieria sygnału – Zmieniamy rzeczywistość własnym kodem!

Do tej pory używaliśmy Twojej autorskiej implementacji Transformacji Fouriera (FFT) jako narzędzia analitycznego. Patrzyliśmy na wykres częstotliwości, aby zobaczyć piki dla 24 i 168 godzin. Ale prawdziwa potęga tej matematyki kryje się w czymś, co nazywamy **Odwrotną Dyskretną Transformatą Fouriera (IDFT / IFFT)**.



### Jak to działa? (Analogia)
Wyobraź sobie, że Twoje FFT to maszyna, która bierze gotowy koktajl owocowy (sygnał zużycia prądu) i rozdziela go z powrotem na pojedyncze składniki: podaje Ci listę, że w środku jest 300g truskawek, 100g bananów i 50g lodu (to są Twoje częstotliwości i amplitudy).

**Odwrotna Transformata (IFFT)** to blender. Wrzucasz do niego z powrotem te składniki, a on miksuje je, odtwarzając dokładnie ten sam, oryginalny koktajl.

Matematycznie, IFFT sumuje z powrotem wszystkie sinusoidalne fale składowe, odtwarzając sygnał w dziedzinie czasu. Opisuje to ten elegancki wzór:

$$x_n = \frac{1}{N} \sum_{k=0}^{N-1} X_k e^{i 2\pi k n / N}$$

Gdzie:
* $X_k$ to nasze "składniki" wyliczone przez Twoje FFT.
* $x_n$ to nasz odtworzony sygnał w dziedzinie czasu (wartości w MW).

### Twój własny filtr dolnoprzepustowy (Low-Pass Filter)
Skoro potrafimy już rozłożyć sygnał własnym kodem, możemy **zmodyfikować składniki przed ich ponownym zmiksowaniem**. Zrobimy teraz coś niesamowitego:
1. Rozłożymy zużycie prądu za pomocą Twojej funkcji `moja_fft`.
2. **Wyzerujemy** wszystkie szybkie, gwałtowne zmiany – czyli cykle dobowe, tygodniowe i "szum" świąteczny.
3. Złożymy sygnał z powrotem za pomocą Twojej funkcji `moja_ifft`.

W efekcie otrzymamy gładką falę pokazującą wyłącznie długoterminowe trendy gospodarcze i sezonowe, a Ty zrobisz to w 100% autorskim silnikiem matematycznym!

## Część 5A: Budujemy własne `moja_ifft` (Matematyczny Hack)

Skoro masz już działającą funkcję `moja_fft(x)`, przyszedł czas na stworzenie jej lustrzanego odbicia – `moja_ifft(X)`. Czy musimy od nowa pisać całą skomplikowaną logikę i dzielić tablicę na części parzyste i nieparzyste? **Absolutnie nie.**

### Matematyczne śledztwo
Porównajmy wzory na klasyczne FFT ($X_k$) oraz odwrotne IFFT ($x_n$):

**FFT (w stronę częstotliwości):**
$$X_k = \sum_{n=0}^{N-1} x_n e^{-i \frac{2\pi}{N} k n}$$

**IFFT (powrót do czasu):**
$$x_n = \frac{1}{N} \sum_{k=0}^{N-1} X_k e^{+i \frac{2\pi}{N} k n}$$

Zwróć uwagę na różnice. Są tylko dwie:
1. W IFFT wykładnik liczby $e$ ma **znak plus** zamiast minusa.
2. Na samym końcu wynik musimy podzielić przez $N$ (długość sygnału).

### Trik ze sprzężeniem zespolonym (Complex Conjugate)
W matematyce zmiana znaku przy części urojonej (czyli przy naszym $i$) to nic innego jak **sprzężenie zespolone**. W bibliotece NumPy wykonujemy je niezwykle prosto, wywołując metodę `.conjugate()` na tablicy.

Dzięki temu możemy "oszukać" naszą funkcję `moje_fft` i zrealizować IFFT w 4 prostych krokach:
1. Weź wejściowe widmo i oblicz jego sprzężenie (odwróć znaki części urojonej).
2. Przepuść ten zmodyfikowany sygnał przez Twoje standardowe `moje_fft`.
3. Wynik z powrotem sprzęgnij zespolenie (ponownie odwróć znaki).
4. Podziel całość przez liczbę próbek $N$.

Ta tożsamość matematyczna gwarantuje, że otrzymasz idealny algorytm odwrotny, dopisując zaledwie kilka linijek kodu!

---

### Twoje zadanie:
W poniższej komórce z kodem znajduje się szablon funkcji `moje_ifft(widmo)`. Uzupełnij go zgodnie z powyższym algorytmem. Na końcu uruchom test sprawdzający, czy sygnał odtworzony zgadza się z oryginałem.

In [ ]:
import numpy as np

def moja_ifft(widmo):
    """
    Autorska implementacja Odwrotnej Transformaty Fouriera
    wykorzystująca trik sprzężenia zespolonego i istniejącą funkcję moje_fft.
    """
   #TODO

# --- TEST POPRAWNOŚCI ---
print("Uruchamianie testu...")

# Tworzymy losowy mały sygnał testowy (długość musi być potęgą 2)
test_sygnal = [1.0, 2.0, 3.0, 4.0]

try:
    # 1. Idziemy w stronę częstotliwości Twoim kodem
    test_widmo = moja_fft(test_sygnal)

    # 2. Wracamy do czasu Twoim nowym kodem
    odtworzony_sygnal = moja_ifft(test_widmo)

    # Wyciągamy część rzeczywistą do porównania
    print(f"Oryginał:   {test_sygnal}")
    print(f"Odtworzony: {np.real(odtworzony_sygnal).tolist()}")

    # Sprawdzamy, czy wartości są bliskie
    if np.allclose(test_sygnal, np.real(odtworzony_sygnal)):
        print("\n✅ SUKCES! Twoje autorskie IFFT działa bezbłędnie!")
    else:
        print("\n❌ Błąd. Wartości się nie zgadzają. Sprawdź swój kod.")

except NameError:
    print("\n❌ BŁĄD: Funkcja 'moje_fft' nie została znaleziona. Upewnij się, że uruchomiłeś komórkę z jej definicją.")

## Część 5B: Polowanie na cykl roczny – Globalny filtr dolnoprzepustowy

Przejdźmy teraz z mikroskali (kilku tygodni) do makroskali. Zamiast tylko usuwać anomalie świąteczne, użyjemy Twojego silnika DSP (`moje_fft` i `moje_ifft`) do odizolowania **czystego cyklu rocznego (sezonowego)** na przestrzeni kilku lat.

Chcemy zobaczyć, jak system energetyczny "oddycha" w zależności od pór roku (ogrzewanie zimą, klimatyzacja latem), całkowicie ignorując szum dni, tygodni, a nawet pojedynczych miesięcy.

**Twoje zadanie - Krok po Kroku:**

1. **Skalowanie do potęgi dwójki ($2^{15}$):** Nasz algorytm FFT wymaga długości sygnału będącej potęgą liczby 2. Aby zbadać około 4 lata danych, dotniemy nasz zbiór do wielkości **$N = 32768$** wierszy ($2^{15} = 32768$, co odpowiada ok. 3,74 roku). Wyciągnij ten fragment z głównej ramki danych.
2. **Transformata Fouriera (FFT):** Przekształć ten długi, kilkuletni sygnał w dziedzinę częstotliwości za pomocą swojej funkcji `moje_fft`. Wygeneruj oś częstotliwości za pomocą `np.fft.fftfreq(N, d=1)`.
3. **Ekstremalne cięcie (Maska filtra):** Tym razem nasz filtr dolnoprzepustowy ustawimy bardzo agresywnie. Chcemy wyciąć z sygnału wszystko, co powtarza się częściej niż raz na **60 dni** (częstotliwości krótkoterminowe, miesięczne, tygodniowe i dobowe idą do kosza). Oblicz próg: `1.0 / (60 * 24)`. Wyzeruj w widmie wszystkie częstotliwości powyżej tego progu.
4. **Rekonstrukcja (IFFT):** Przywróć przefiltrowane widmo do świata czasu za pomocą `moje_ifft`. Wyciągnij część rzeczywistą (`np.real`) i zapisz ją jako nową kolumnę (np. `Trend_Roczny`).
5. **Wizualizacja:** Narysuj na jednym wykresie cały przygotowany segment danych (wszystkie 32768 godzin!). Zobaczysz, jak pomarańczowa linia Twojego filtra idealnie odwzorowuje kilkuletnią, gładką sinusoidę pór roku.

### Instrukcja krok po kroku – jak napisać kod:

#### Krok 1: Przygotowanie wieloletniego sygnału
Wydajne algorytmy FFT wymagają, aby długość danych była potęgą dwójki. Najbliższą potęgą odpowiadającą około 4 latom jest **$2^{15} = 32768$** godzin (ok. 3,74 roku).
* Wyetapuj z głównej ramki danych `df` pierwsze 32768 wierszy za pomocą `.iloc[:32768]` i stwórz kopię (`.copy()`).
* Wyciągnij wartości z kolumny `Zuzycie_MW` jako czystą tablicę NumPy (`.values`).

#### Krok 2: Rozkład sygnału (FFT) oraz oś częstotliwości
* Przekaż przygotowaną tablicę sygnału do swojej funkcji `moje_fft()`. Pamiętaj, aby owinąć wynik w `np.array()`.
* Wygeneruj wektor częstotliwości za pomocą `np.fft.fftfreq(N, d=1)`. Parametr `d=1` oznacza krok próbkowania co 1 godzinę.

#### Krok 3: Agresywne filtrowanie widma
Chcemy zbudować filtr, który zablokuje wszelkie zmiany zachodzące częściej niż raz na **60 dni** (2 miesiące).
* Stwórz bezpieczną kopię widma za pomocą `.copy()`.
* Oblicz próg godzinowy: `60 * 24`.
* Oblicz próg częstotliwości: `1.0 / prog_godzinowy`.
* Wykorzystaj maskowanie logiczne NumPy. Znajdź miejsca, gdzie wartość bezwzględna częstotliwości (`np.abs(czestotliwosci)`) jest **większa** niż próg, i przypisz tam wartość `0`.

#### Krok 4: Odtwarzanie trendu (IFFT)
* Przekaż przefiltrowane widmo do swojej nowo napisanej funkcji `moje_ifft()`.
* Ponieważ procesor generuje minimalne błędy zaokrągleń w części urojonej, odetnij ją, używając `np.real()`.
* Zapisz uzyskany wynik jako nową kolumnę `'Trend_Roczny'` w swojej wyciętej ramce danych.

#### Krok 5: Wizualizacja makroskopowa
* Stwórz wykres za pomocą `plt.figure(figsize=(15, 6))`.
* Narysuj oryginalne dane godzinowe. Użyj jasnego, szarego koloru (`color='lightgray'`) i lekkiej przezroczystości (`alpha=0.6`), aby stanowiły tło.
* Nałóż na to linię trendu rocznego. Użyj kontrastowego koloru (np. `'crimson'` lub `'darkorange'`) i zwiększ grubość linii (`linewidth=3`).
* Dodaj legendę, tytuł wykresu oraz siatkę (`plt.grid`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("Uruchamiam globalny filtr sezonowy...")

# --- KROK 1: Przygotowanie wieloletniego sygnału (2^15 = 32768) ---
N = 32768

# Wyciągamy pierwsze N wierszy z df i robimy kopię
df_wieloletni = df.iloc[:N].copy()

# Wyciągamy kolumnę 'Zuzycie_MW' jako tablicę numpy
sygnal_wieloletni = df_wieloletni['Zuzycie_MW'].values


# --- KROK 2: Rozkład wielkiego sygnału (FFT) ---
print("Obliczam FFT dla 32768 godzin danych...")

# Używamy autorskiej funkcji moje_fft
widmo_wieloletnie = np.array(moja_fft(sygnal_wieloletni))

# Generujemy wektor częstotliwości (krok próbkowania d=1 godzina)
czestotliwosci = np.fft.fftfreq(N, d=1)


# --- KROK 3: Agresywne filtrowanie (Odcięcie cykli szybszych niż 60 dni) ---
widmo_przefiltrowane = widmo_wieloletnie.copy()

# Obliczamy próg w godzinach i częstotliwość odcięcia
prog_godzinowy = 60 * 24
prog_czestotliwosci = 1.0 / prog_godzinowy

# Zerujemy wysokie częstotliwości w widmo_przefiltrowane
widmo_przefiltrowane[np.abs(czestotliwosci) > prog_czestotliwosci] = 0


# --- KROK 4: Rekonstrukcja wieloletniego trendu (IFFT) ---
print("Rekonstruuję czysty cykl roczny za pomocą moje_ifft...")

# Przekazujemy przefiltrowane widmo do autorskiej funkcji moje_ifft
sygnal_roczny_zespolony = moja_ifft(widmo_przefiltrowane)

# Wyciągamy część rzeczywistą za pomocą np.real i zapisujemy do ramki danych
df_wieloletni['Trend_Roczny'] = np.real(sygnal_roczny_zespolony)


# --- KROK 5: Wizualizacja makroskopowa ---
print("Generuję wykres wieloletni...")
plt.figure(figsize=(15, 6))

# Rysujemy oryginalne zużycie (jasnoszary kolor, lekka przezroczystość)
plt.plot(df_wieloletni.index, df_wieloletni['Zuzycie_MW'], color='lightgray', alpha=0.6, label='Oryginalne zużycie (szum dobowy/tygodniowy)')

# Rysujemy przefiltrowany trend (gruba linia, wyraźny kolor)
plt.plot(df_wieloletni.index, df_wieloletni['Trend_Roczny'], color='crimson', linewidth=3, label='Czysty cykl roczny (Pory roku)')

# Formatowanie wykresu
plt.title('Makroskopowy cykl roczny wyekstrahowany autorskim systemem FFT/IFFT', fontsize=14)
plt.xlabel('Data')
plt.ylabel('Zużycie energii [MW]')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("\nProces zakończony. Sprawdź, czy na wykresie wyłoniła się gładka, wieloletnia sinusoida!")